# 🏋️ PulseFit · Notebook 05
## Snowflake superpowers: a very bad Monday

| 01 Set up | 02 Bronze | 03 Silver & Gold | 04 Answer the CEO | 05 Superpowers |
|:-:|:-:|:-:|:-:|:-:|
| ✅ done | ✅ done | ✅ done | ✅ done | 📍 **you are here** |

### Where we are
You answered the CEO. 🎉 Now it's Monday morning at PulseFit, and things start going wrong. In this notebook you'll
use the features engineers love most (lecture: *"Features engineers love"* and *"Beyond the warehouse"*):

| Problem | Superpower |
|---|---|
| 😱 Someone deleted data by mistake | **Time Travel** |
| 😱 Someone dropped a whole table | **UNDROP** |
| 🧪 Finance wants to test a change without touching production | **Zero-copy cloning** |
| 🔐 Analysts should see gold, but not raw data | **Role-based access control** |
| 🤝 A partner wants our data | **Secure data sharing** and the **Marketplace** |
| 💰 "So what did all this cost?" | **Usage and storage views** |

## Step 0: Set the context
> If you see *"does not exist or not authorized"*, run notebooks **01** to **03** first.

In [ ]:
%%sql -r context
-- Every notebook starts by telling Snowflake WHO we are and WHERE we work
USE ROLE ACCOUNTADMIN;
USE WAREHOUSE PULSEFIT_WH;
USE DATABASE PULSEFIT_DW;
USE SCHEMA GOLD;

SELECT CURRENT_ROLE()      AS my_role,
       CURRENT_WAREHOUSE() AS my_warehouse,
       CURRENT_DATABASE()  AS my_database,
       CURRENT_SCHEMA()    AS my_schema;

In [ ]:
from snowflake.snowpark.context import get_active_session

session = get_active_session()
print("Connected as", session.get_current_role())

## Step 1: 😱 "I deleted the Colombo visits!" → Time Travel
An intern wanted to delete a few test rows for the Colombo club, but the filter was wrong, and **every Colombo
visit** is about to disappear from the fact table. First, how many rows do we have now?

In [ ]:
%%sql -r rows_before_mistake
SELECT COUNT(*)                                     AS all_visits,
       COUNT_IF(c.club_code = 'CMB')                AS colombo_visits
FROM GOLD.FCT_CHECKIN f
JOIN GOLD.DIM_CLUB c ON c.club_key = f.club_key;

Now the mistake. We also save the **query ID** of the bad statement in a SQL variable (`$oops_query_id`), so we can
travel back to the moment just before it.

> ⚠️ **Run this cell only once.** (If you accidentally run it twice, just re-run notebook 03 to rebuild gold.
> Being able to rebuild everything from bronze is exactly why we kept bronze!)

In [ ]:
%%sql -r the_mistake
DELETE FROM GOLD.FCT_CHECKIN
WHERE club_key = (SELECT club_key FROM GOLD.DIM_CLUB WHERE club_code = 'CMB');

SET oops_query_id = LAST_QUERY_ID();

SELECT $oops_query_id                          AS oops_query_id,
       (SELECT COUNT(*) FROM GOLD.FCT_CHECKIN) AS visits_left;

Every change in Snowflake writes **new** micro-partitions; the old ones are kept for a while. That's how
**Time Travel** can show a table **as it was in the past**:
* `BEFORE (STATEMENT => '<query id>')`: just before a specific statement
* `AT (OFFSET => -600)`: 10 minutes ago
* `AT (TIMESTAMP => '2026-07-01 09:00'::TIMESTAMP_LTZ)`: at an exact time

In [ ]:
%%sql -r time_travel_look
SELECT 'now'                AS version, COUNT(*) AS row_count FROM GOLD.FCT_CHECKIN
UNION ALL
SELECT 'before the mistake' AS version, COUNT(*) AS row_count
FROM GOLD.FCT_CHECKIN BEFORE (STATEMENT => $oops_query_id);

The deleted rows still exist in the past. Let's copy them back. The `NOT IN` makes this restore **rerunnable**:
running it twice can't create duplicates.

In [ ]:
%%sql -r time_travel_restore
INSERT INTO GOLD.FCT_CHECKIN
SELECT *
FROM GOLD.FCT_CHECKIN BEFORE (STATEMENT => $oops_query_id)
WHERE checkin_id NOT IN (SELECT checkin_id FROM GOLD.FCT_CHECKIN);

SELECT COUNT(*)                      AS all_visits,
       COUNT_IF(c.club_code = 'CMB') AS colombo_visits
FROM GOLD.FCT_CHECKIN f
JOIN GOLD.DIM_CLUB c ON c.club_key = f.club_key;

👀 Same numbers as before the mistake. No backup tapes, no ticket to the DBA team, about one minute of work.

## Step 2: 😱 "I dropped a table!" → UNDROP
Dropping a table doesn't destroy it straight away either. It stays recoverable for the table's
**retention period**:

In [ ]:
%%sql -r retention_period
SHOW PARAMETERS LIKE 'DATA_RETENTION_TIME_IN_DAYS' IN TABLE GOLD.DIM_PLAN;

👀 `1` day is the default. On the Enterprise edition (which trial accounts use) you can raise it to up to 90 days
per table, schema or database.

Now drop the plan dimension (on purpose!) and check that it's gone:

In [ ]:
%%sql -r drop_table
DROP TABLE GOLD.DIM_PLAN;

SHOW TABLES LIKE 'DIM_PLAN' IN SCHEMA GOLD;

👀 An empty result: the table is gone. Our analysts' view `CHECKIN_DETAILS` is broken too. Let's bring it back:

> ⚠️ Run this cell only once (UNDROP fails if the table already exists).

In [ ]:
%%sql -r undrop_table
UNDROP TABLE GOLD.DIM_PLAN;

SELECT * FROM GOLD.DIM_PLAN ORDER BY plan_key;

## Step 3: 🧪 "Can I test a change without touching production?" → zero-copy cloning
Finance wants to try out *"what if Premium cost LKR 15,000?"*, but nobody should touch production.

A **clone** is a full copy of a table, schema or **whole database**, created in seconds. It costs **no extra
storage** at first, because it points at the same micro-partitions. Only what you change afterwards is stored
separately.

In [ ]:
%%sql -r clone_database
CREATE OR REPLACE DATABASE PULSEFIT_DEV CLONE PULSEFIT_DW;

-- CREATE DATABASE switches us into the new database, so switch back to production
USE DATABASE PULSEFIT_DW;
USE SCHEMA GOLD;

SELECT 'PULSEFIT_DW  (production)' AS database_name, COUNT(*) AS fct_checkin_rows FROM PULSEFIT_DW.GOLD.FCT_CHECKIN
UNION ALL
SELECT 'PULSEFIT_DEV (clone)',                        COUNT(*)                    FROM PULSEFIT_DEV.GOLD.FCT_CHECKIN;

👀 A complete copy of the warehouse (every layer, every table) in a second or two. Now change the price **in the
clone only** and compare:

In [ ]:
%%sql -r change_dev_only
UPDATE PULSEFIT_DEV.GOLD.DIM_PLAN
SET monthly_fee_lkr = 15000
WHERE plan_code = 'PREMIUM' AND is_current;

SELECT 'production' AS environment, plan_code, monthly_fee_lkr
FROM PULSEFIT_DW.GOLD.DIM_PLAN  WHERE plan_code = 'PREMIUM' AND is_current
UNION ALL
SELECT 'dev clone',                 plan_code, monthly_fee_lkr
FROM PULSEFIT_DEV.GOLD.DIM_PLAN WHERE plan_code = 'PREMIUM' AND is_current;

👀 Production still says 14,000; the clone says 15,000. Real teams use clones for **dev and test environments**,
for testing pipeline changes on real data, and for safe "what-if" analysis (lecture: *"One codebase, three
environments"*).

## Step 4: 🔐 "Analysts should only see gold" → role-based access control
In Snowflake, **privileges are granted to roles, and roles are granted to users**. Let's create a
`PULSEFIT_ANALYST` role that can use the warehouse and read **gold only**:

In [ ]:
%%sql -r create_analyst_role
CREATE ROLE IF NOT EXISTS PULSEFIT_ANALYST COMMENT = 'Can read the GOLD layer only';

GRANT USAGE  ON WAREHOUSE PULSEFIT_WH               TO ROLE PULSEFIT_ANALYST;
GRANT USAGE  ON DATABASE  PULSEFIT_DW               TO ROLE PULSEFIT_ANALYST;
GRANT USAGE  ON SCHEMA    PULSEFIT_DW.GOLD          TO ROLE PULSEFIT_ANALYST;
GRANT SELECT ON ALL TABLES IN SCHEMA PULSEFIT_DW.GOLD TO ROLE PULSEFIT_ANALYST;
GRANT SELECT ON ALL VIEWS  IN SCHEMA PULSEFIT_DW.GOLD TO ROLE PULSEFIT_ANALYST;

-- Best practice: custom roles roll up to SYSADMIN. Then give the role to yourself so you can test it.
GRANT ROLE PULSEFIT_ANALYST TO ROLE SYSADMIN;
SET my_user = '"' || CURRENT_USER() || '"';
GRANT ROLE PULSEFIT_ANALYST TO USER IDENTIFIER($my_user);

SHOW GRANTS TO ROLE PULSEFIT_ANALYST;

Now **become an analyst** for a moment and try to read gold and bronze. (Your session normally has all your
roles active at once as *secondary roles*, so we switch those off to see only what the analyst role allows.)

In [ ]:
def try_to_read(table):
    try:
        n = session.sql(f"SELECT COUNT(*) FROM {table}").collect()[0][0]
        return f"✅ allowed: {n:,} rows"
    except Exception as e:
        reason = str(e).strip().splitlines()[-1].split(". ")[0]
        return "⛔ blocked: " + reason

access_check = {}
try:
    session.sql("USE ROLE PULSEFIT_ANALYST").collect()
    session.sql("USE SECONDARY ROLES NONE").collect()
    access_check["working as"] = session.get_current_role().strip('"')
    access_check["GOLD.FCT_CHECKIN"] = try_to_read("PULSEFIT_DW.GOLD.FCT_CHECKIN")
    access_check["BRONZE.CHECKIN_EVENTS"] = try_to_read("PULSEFIT_DW.BRONZE.CHECKIN_EVENTS")
finally:
    session.sql("USE ROLE ACCOUNTADMIN").collect()
    session.sql("USE SECONDARY ROLES ALL").collect()

for what, outcome in access_check.items():
    print(f"{what:<22} → {outcome}")
print("Back to:", session.get_current_role().strip('"'))

👀 The analyst can read gold, while bronze is *"does not exist or not authorized"*. Snowflake doesn't even reveal
that the table exists. One governed copy of the data, with different access for each role.

## Step 5: 🤝 "Our partner wants our data" → secure data sharing
With **secure data sharing**, one Snowflake account gives another account **live, read-only access** to
tables, with **no copying, no files and no FTP**. The consumer always sees the latest data.

You have already used a share. `SNOWFLAKE_SAMPLE_DATA`, the 600-million-row database from notebook 04, is shared
with your account by Snowflake. It takes **zero storage** in your account:

In [ ]:
%%sql -r shared_databases
SHOW DATABASES;

SELECT "name" AS database_name, "kind", "origin" AS shared_from
FROM TABLE(RESULT_SCAN(LAST_QUERY_ID()))
ORDER BY "kind", "name";

👀 Look at the `kind` column. `IMPORTED DATABASE` means it comes **from a share** (`shared_from` tells you who
shared it). `STANDARD` databases are your own. Your `PULSEFIT_DEV` clone is there too.

Sharing works in the other direction as well. Here's the **provider side**: a share that offers PulseFit's
club list to a partner (for example a corporate-wellness company):

In [ ]:
%%sql -r create_share
CREATE SHARE IF NOT EXISTS PULSEFIT_PARTNER_SHARE COMMENT = 'PulseFit club list for partners';

GRANT USAGE  ON DATABASE PULSEFIT_DW               TO SHARE PULSEFIT_PARTNER_SHARE;
GRANT USAGE  ON SCHEMA   PULSEFIT_DW.GOLD          TO SHARE PULSEFIT_PARTNER_SHARE;
GRANT SELECT ON TABLE    PULSEFIT_DW.GOLD.DIM_CLUB TO SHARE PULSEFIT_PARTNER_SHARE;

SHOW GRANTS TO SHARE PULSEFIT_PARTNER_SHARE;

To actually deliver it, you would run `ALTER SHARE PULSEFIT_PARTNER_SHARE ADD ACCOUNTS = <partner_account>;`.
The partner would then see `DIM_CLUB` as a live, read-only database. We stop here because we have no partner
account.

🛒 **Snowflake Marketplace** is built on exactly this mechanism: thousands of datasets (weather, holidays,
demographics, finance, ...) that you can add to your account in a few clicks, with no copying. **Try it later:**
in Snowsight, open the **Marketplace**, search for a free dataset such as *weather*, select **Get**, and it appears
as a new database within seconds.

## Step 6: 💰 "What did all this cost?"
**Compute**: credits used by each warehouse in the last day.
> The numbers can lag by a few minutes, so very recent queries may not show yet.

In [ ]:
%%sql -r compute_credits
SELECT warehouse_name,
       ROUND(SUM(credits_used), 3) AS credits_used
FROM TABLE(PULSEFIT_DW.INFORMATION_SCHEMA.WAREHOUSE_METERING_HISTORY(
         DATE_RANGE_START => DATEADD(day, -1, CURRENT_TIMESTAMP())))
GROUP BY warehouse_name
ORDER BY credits_used DESC;

**Storage**: how much space each of our tables takes, after Snowflake's automatic compression:

In [ ]:
%%sql -r storage_used
SELECT table_schema,
       table_name,
       row_count,
       ROUND(bytes / 1024 / 1024, 2) AS size_mb
FROM PULSEFIT_DW.INFORMATION_SCHEMA.TABLES
WHERE table_schema IN ('BRONZE', 'SILVER', 'GOLD')
  AND table_type = 'BASE TABLE'
ORDER BY table_schema, size_mb DESC;

👀 **What to notice**
* The whole practical used **well under one credit** of warehouse compute. A credit costs a few US dollars,
  depending on edition and region, and your trial came with free credits.
* The check-in JSON (about 50 MB raw) takes only a few MB in Snowflake. Storage is billed per compressed
  terabyte per month, so keeping years of history is cheap.
* The notebook itself also runs on a small container compute service, which is billed separately while it runs.
  See *Before you leave* below.

## 🏁 The end of the journey
| Lecture idea | Where you used it |
|---|---|
| Storage and compute are separate | created `PULSEFIT_WH` in a second, resized it to Medium and back (01, 04) |
| Bronze ▸ Silver ▸ Gold, layered architecture | `PULSEFIT_DW` schemas (01–03) |
| ELT: load raw, transform inside | `COPY INTO` as-is (02), SQL transformations (03) |
| Semi-structured data | JSON in `VARIANT`, `:` paths, `FLATTEN` (02) |
| Rerunnable pipelines | `COPY INTO` loaded nothing twice (02), rerunnable restore (05) |
| Integrated, time-variant | time zones, card ➜ member, SCD Type 2 history, price validity windows (03) |
| Star schema: grain, facts, dimensions, surrogate keys | `FCT_CHECKIN` and four dimensions (03) |
| Data quality as a gate | tests that stop the pipeline (03) |
| Micro-partitions, pruning, caching, elastic compute | 600 million rows (04) |
| Time Travel, UNDROP, zero-copy cloning | a very bad Monday (05) |
| Governance and sharing | analyst role, secure share, Marketplace (05) |

### What we did *not* cover (good next steps)
* **Snowpipe**: load files automatically the moment they arrive.
* **Streams, Tasks and Dynamic Tables**: run pipelines on a schedule or keep tables fresh automatically.
* **dbt**: manage the "T" as tested, version-controlled code (lecture: *"The tool for the T"*).
* **Cortex AI**: call large language models straight from SQL.
* **Snowflake Quickstarts** (free step-by-step labs on the Snowflake developer site).

## 🧹 Before you leave
1. **The warehouse** suspends by itself after 60 seconds, so nothing to do there.
2. **Stop your notebook service** (the container compute behind this notebook) from the notebook's connection
   menu. Otherwise it keeps running until its idle timeout.
3. Want to remove everything you built? Run **`99_Cleanup_Optional`**.

## 🤔 Final questions
1. Which of today's features would have saved the most time or money in a project you know (or can imagine)?
2. PulseFit now has a warehouse that answers one question. What would you need to add so it refreshes **every
   night, automatically**, and alerts someone when a quality test fails?

**Congratulations, you've built a data warehouse in Snowflake! 🎉**